# Chapter 1
## Fig 1.7

Collins toggle switch model for Figures 1.7, 7.13, 7.14, 7.15.

In [1]:
using ModelingToolkit
using ModelingToolkit: t_nounits as t, D_nounits as D
using OrdinaryDiffEq
using Plots
Plots.gr(linewidth=1.5)

Plots.GRBackend()

In [2]:
function collins_sys(; name=:collins)
    @parameters a1=3 a2=2.5 β=4 γ=4
    @variables i1(t) i2(t) s1(t)=0.075 s2(t)=2.5
    hil(x, k) = x / (x + k)
    hil(x, k, n) = hil(x^n, k^n)
    eqs = [
        D(s1) ~ a1 * hil(1 + i2, s2, β) - s1,
        D(s2) ~ a2 * hil(1 + i1, s1, γ) - s2,
        i1 ~ 10 * (t > 30) * (t < 40),
        i2 ~ 10 * (t > 10) * (t < 20)
    ]
    return System(eqs, t; name)
end

collins_sys (generic function with 1 method)

In [4]:
tend = 50.0
@time "Build system" @mtkbuild sys = collins_sys()
@time "Build problem" prob = ODEProblem(sys, [], tend)
@time "Solve problem" sol = solve(prob, Tsit5(), tstops=[10.0, 20.0, 30.0, 40.0])

Build system: 0.198572 seconds (8.36 k allocations: 373.702 KiB)
Build problem: 0.004979 seconds (27.31 k allocations: 1.415 MiB)
Solve problem: 0.000275 seconds (1.23 k allocations: 62.914 KiB)


retcode: Success
Interpolation: specialized 4th order "free" interpolation
t: 51-element Vector{Float64}:
  0.0
  0.38548049109061366
  1.2786313586853497
  2.464538731594535
  4.06089298356771
  6.199065127274195
  9.163107137014421
 10.0
 10.237564173675388
 10.345905622741512
  ⋮
 36.914997164885754
 37.93079710385752
 39.07484274669089
 40.0
 41.178943649875706
 42.81986325559389
 44.70418578722446
 47.243347624826804
 50.0
u: 51-element Vector{Vector{Float64}}:
 [2.5, 0.075]
 [2.4999747263084027, 0.07496310584167311]
 [2.499943104158718, 0.07491895384143081]
 [2.499927975740838, 0.07489947360105878]
 [2.4999227222886917, 0.07489347508722309]
 [2.499921565722204, 0.07489238103870029]
 [2.4999214349680665, 0.0748923054927769]
 [2.4999213893330796, 0.07489223296465887]
 [2.372119211508154, 0.6391346146761684]
 [2.3191402463267354, 0.8809050681389737]
 ⋮
 [2.4974663084645288, 0.08211174811519019]
 [2.4990818963098618, 0.07761093651936424]
 [2.499707057292954, 0.07579211279532862]
 [2.

In [ ]:
plot(sol, title="Fig. 1.7", xlabel="Time", ylabel="Concentration")

In [ ]:
plot(sol, idxs=[sys.i1, sys.i2], labels=["i1" "i2"], xlabel="Time", ylabel="Signal strength")

## Fig 1.09

Hodgkin-Huxley model

In [ ]:

using OrdinaryDiffEq
using ModelingToolkit
using ModelingToolkit: t_nounits as t, D_nounits as D
using Plots
Plots.gr(linewidth=1.5)

In [ ]:
function hh_sys(; name=:hh)
    exprel(x) = x / expm1(x)
    @discretes iStim(t)=0.0
    @parameters E_N=55 E_K=-72 E_LEAK=-49 G_N_BAR=120 G_K_BAR=36 G_LEAK=0.3 C_M=1
    @variables v(t)=-59.8977 m(t)=0.0536 h(t)=0.5925 n(t)=0.3192 iNa(t) iK(t) iLeak(t) ma(t) mb(t) ha(t) hb(t) na(t) nb(t)

    ## Electrical stimulation events
    stim_on_1 = ModelingToolkit.SymbolicDiscreteCallback([20.0] => [iStim ~ -6.6], discrete_parameters = iStim, iv = t)
    stim_off_1 = ModelingToolkit.SymbolicDiscreteCallback([21.0] => [iStim ~ 0.0], discrete_parameters = iStim, iv = t)
    stim_on_2 = ModelingToolkit.SymbolicDiscreteCallback([60.0] => [iStim ~ -6.9], discrete_parameters = iStim, iv = t)
    stim_off_2 = ModelingToolkit.SymbolicDiscreteCallback([61.0] => [iStim ~ 0.0], discrete_parameters = iStim, iv = t)

    eqs = [
        ma ~ exprel(-0.10 * (v + 35)),
        mb ~ 4.0 * exp(-(v + 60) / 18.0),
        ha ~ 0.07 * exp(-(v + 60) / 20),
        hb ~ 1 / (exp(-(v + 30) / 10) + 1),
        na ~ 0.1 * exprel(-0.1 * (v + 50)),
        nb ~ 0.125 * exp(-(v + 60) / 80),
        iNa ~ G_N_BAR * (v - E_N) * (m^3) * h,
        iK ~ G_K_BAR * (v - E_K) * (n^4),
        iLeak ~ G_LEAK * (v - E_LEAK),
        D(v) ~ -(iNa + iK + iLeak + iStim) / C_M,
        D(m) ~ -(ma + mb) * m + ma,
        D(h) ~ -(ha + hb) * h + ha,
        D(n) ~ -(na + nb) * n + na
    ]
    sys = ODESystem(eqs, t; name, discrete_events=[stim_on_1, stim_off_1, stim_on_2, stim_off_2])
    return sys
end

In [ ]:
tend = 100.0
@time "Build system" @mtkcompile sys = hh_sys()
@time "Build problem" prob = ODEProblem(sys, [], tend)
@time "Solve problem" sol = solve(prob, TRBDF2())

In [ ]:
plot(sol, idxs=[sys.v], xlabel="Time (ms)", ylabel="Membrane potential (mV)", title="Fig 1.9")